In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install sentence-transformers -q

In [3]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
import os

BASE_PATH = "/content/drive/MyDrive/amazon-customer-review/"

# Amazon Customer Review — Relabeling Pipeline

Bu notebook, Amazon müşteri yorumlarını şikayet kategorilerine göre etiketler.

## Yöntem
1. `review_body` metni SentenceTransformer ile embedding'e çevrilir
2. PCA ile boyut azaltılır (384 -> 50)
3. MiniBatchKMeans ile kümeler oluşturulur
4. Kümelere manuel etiket atanır, kargo/satıcı şikayetleri keyword bazlı eklenir

## Kategoriler ve Departmanlar

| Kategori | Açıklama | Departman |
|----------|----------|-----------|
| `problem_yok` | Şikayet içermeyen yorum | — |
| `ürün_kalitesi` | Fiziksel kalite sorunu (malzeme, yapım hatası) | Kalite Kontrol |
| `ürün_dayanıklılığı` | Zamanla bozulma (2 haftada kırıldı) | Ürün Geliştirme |
| `teknik_sorun` | İşlevsel sorun (çalışmıyor, bağlanmıyor) | Teknik Destek |
| `içerik_beklenti` | Ürün açıklamasıyla uyuşmazlık | Pazarlama |
| `kargo_teslimat` | Kargo/teslimat sorunu | Lojistik |
| `satıcı` | Satıcı kaynaklı sorun | Satıcı İlişkileri |

## Örnek Yorumlar

**ürün_kalitesi:** *"Plastic feels cheap and flimsy"*
**ürün_dayanıklılığı:** *"Broke after 2 weeks of normal use"*
**teknik_sorun:** *"Doesn't connect to Bluetooth, won't turn on"*
**kargo_teslimat:** *"Arrived damaged, box was crushed"*
**satıcı:** *"Seller sent wrong item, no response to emails"*

In [4]:
df = pd.read_csv(BASE_PATH + "data/combined_clean.csv")
print(f"Toplam: {len(df)}")
print(df.columns.tolist())

Toplam: 533680
['review_headline', 'review_body', 'star_rating', 'verified_purchase', 'helpful_votes', 'total_votes']


## Embedding Üretimi (Tekrar Çalıştırılmadı)

Aşağıdaki kod `review_body` metinlerini SentenceTransformer (`all-MiniLM-L6-v2`) ile
384 boyutlu embedding'lere çevirir. 533k satır CPU/GPU'da uzun sürdüğü için
10k'lık batch'ler halinde işlenip `data/batches_v2/` klasörüne kaydedilmiştir.
Bu adım tamamlanmıştır, sonraki hücre kayıtlı batch'leri okur.

# REFERANS — Bu hücre zaten çalıştırıldı, batch'ler data/batches_v2/ klasöründe mevcut
"""
model = SentenceTransformer('all-MiniLM-L6-v2')
BATCH_SIZE = 10000
SAVE_PATH = BASE_PATH + "data/batches_v2/"
os.makedirs(SAVE_PATH, exist_ok=True)

total_batches = (len(df) // BATCH_SIZE) + 1
for i in range(total_batches):
    batch = df.iloc[i*BATCH_SIZE:(i+1)*BATCH_SIZE].copy()
    if len(batch) == 0:
        break
    embeddings = model.encode(batch['review_body'].tolist(), show_progress_bar=False)
    batch['embedding'] = embeddings.tolist()
    batch.to_pickle(SAVE_PATH + f'batch_{i:03d}.pkl')
"""

## Embedding

Embedding'ler `data/batches_v2/` klasöründe batch batch kaydedildi (54 batch, toplam 533680 satır). Aşağıdaki hücre bu batch'leri yükler ve birleştirir.

In [5]:
SAVE_PATH = BASE_PATH + "data/batches_v2/"
batch_files = sorted([f for f in os.listdir(SAVE_PATH) if f.startswith('batch_')])

dfs = []
for f in batch_files:
    df_b = pd.read_pickle(SAVE_PATH + f)
    dfs.append(df_b)

df_all = pd.concat(dfs).reset_index(drop=True)
print("Toplam:", len(df_all))

embeddings = np.array(df_all['embedding'].tolist())
print("Embedding shape:", embeddings.shape)

Toplam: 533680
Embedding shape: (533680, 384)


## Clustering (PCA + KMeans)

384 boyutlu embedding'ler PCA ile 50 boyuta indirilir, sonra MiniBatchKMeans ile 7 kümeye ayrılır.

In [6]:

pca = PCA(n_components=50, random_state=42)
embeddings_reduced = pca.fit_transform(embeddings)
print("Reduced shape:", embeddings_reduced.shape)

kmeans = MiniBatchKMeans(n_clusters=7, random_state=42, batch_size=10000)
kmeans.fit(embeddings_reduced)
df_all['cluster'] = kmeans.labels_
print(df_all['cluster'].value_counts())

for cluster_id in range(7):
    print(f"\n--- Cluster {cluster_id} ---")
    samples = df_all[df_all['cluster'] == cluster_id]['review_body'].sample(5, random_state=42).tolist()
    for s in samples:
        print(f"  • {s[:150]}")

Reduced shape: (533680, 50)
cluster
6    111076
4     93530
1     92140
2     71156
5     63002
3     58338
0     44438
Name: count, dtype: int64

--- Cluster 0 ---
  • I love these headphones, and they are sold at a great price.<br />I find them to fit very comfortably and have superb sound.
  • Pretty good, my ears get tired if I wear too long but I am happy with them.
  • Worked great for about 2 weeks. Would not bother buying another pair. It's a shame since my earphone ones lasted for a couple of years. Very disappoin
  • I gave these generic ones a try. Sound quality, fit, comfort, and battery are decent. However none of that matters if the Bluetooth conection sucks. W
  • These earbuds became one of the most disappointing purchases that I have ever made here on Amazon. When I received them I was under the impression tha

--- Cluster 1 ---
  • Buttons don't work properly
  • Works great! No issues with the connection whatsoever.
  • After 12 hours of charging, it still shows 3 ba

## Etiket Atama (7 cluster + keyword)

7 cluster içinde `ürün_dayanıklılığı` ve `içerik_beklenti` net ayrışmadı, bu yüzden bu iki kategori için de keyword bazlı kurallar ekliyoruz.

In [7]:
# Cluster -> temel etiket
cluster_label_map = {
    0: 'problem_yok',
    1: 'teknik_sorun',
    2: 'problem_yok',
    3: 'ürün_kalitesi',
    4: 'problem_yok',
    5: 'problem_yok',
    6: 'problem_yok'
}

df_all['base_label'] = df_all['cluster'].map(cluster_label_map)

# Keyword listeleri
kargo_keywords = ['shipping', 'delivery', 'arrived', 'package', 'courier', 'shipped', 'transit', 'usps', 'fedex', 'ups', 'damaged in']
satici_keywords = ['seller', 'vendor', 'sold by', 'third party', 'marketplace', 'merchant', 'wrong item', 'not as advertised', 'fake', 'counterfeit']
durability_keywords = ['broke after', 'stopped working after', 'died after', 'wore out', 'fell apart', 'lasted only', 'months later', 'weeks later', 'no longer works', 'quit working']
expectation_keywords = ['not what i expected', 'different from', 'not as described', 'smaller than expected', 'larger than expected', 'not as pictured', 'misleading']

def assign_label(row):
    text = str(row['review_body']).lower()
    star = row['star_rating']

    if star <= 2:
        if any(k in text for k in kargo_keywords):
            return 'kargo_teslimat'
        if any(k in text for k in satici_keywords):
            return 'satıcı'
        if any(k in text for k in durability_keywords):
            return 'ürün_dayanıklılığı'
        if any(k in text for k in expectation_keywords):
            return 'içerik_beklenti'

    return row['base_label']

df_all['problem_category'] = df_all.apply(assign_label, axis=1)

print(df_all['problem_category'].value_counts())

problem_category
problem_yok           376495
teknik_sorun           87295
ürün_kalitesi          56938
kargo_teslimat          6303
ürün_dayanıklılığı      3120
satıcı                  2883
içerik_beklenti          646
Name: count, dtype: int64


## Eski Etiketli Veri ile Birleştirme

In [9]:
df_final = df_all.drop(columns=['embedding', 'cluster', 'base_label'])
print(df_final['problem_category'].value_counts())

df_final.to_csv(BASE_PATH + "data/labeled_data_full.csv", index=False)
print("Kaydedildi.")

problem_category
problem_yok           376495
teknik_sorun           87295
ürün_kalitesi          56938
kargo_teslimat          6303
ürün_dayanıklılığı      3120
satıcı                  2883
içerik_beklenti          646
Name: count, dtype: int64
Kaydedildi.
